# Policy Gradient

**Reinforcement Learning Course** — Notebook 04

| Algorithm | Environment | Key idea |
|-----------|-------------|----------|
| REINFORCE | Hopper-v4 | Monte Carlo PG |
| PPO | Hopper-v4 | Clipped surrogate · vectorised envs |

In [ ]:
%pip install -q "gymnasium[mujoco]" torch plotly matplotlib xvfbwrapper
import os
os.environ["MUJOCO_GL"] = "egl"

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Part 1 — REINFORCE

**Policy Gradient Theorem:**

$$\nabla_\theta J(\theta) = \underset{\substack{S_t \sim \rho^{\pi_\theta}_\gamma \\ A_t \sim \pi_\theta(\cdot \mid S_t)}}{\mathbb{E}}\!\left[\nabla_\theta \log \pi_\theta(A_t \mid S_t)\cdot G_t\right]$$

For continuous actions: $\pi_\theta(\cdot \mid s) = \mathcal{N}(\mu_\theta(s),\, e^{2\sigma})$, $\sigma$ a learned log-std.  
Rollout → compute $G_t$ → gradient step. One episode at a time.

In [ ]:
# ── Policy network (MLP → Gaussian mean) ──────────────────────────────────────
class PolicyNet(nn.Module):
    def __init__(self, obs_dim: int, act_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.ReLU(),
            nn.Linear(64, 64),      nn.ReLU(),
            nn.Linear(64, act_dim),
        )
        self.log_std = nn.Parameter(torch.full((act_dim,), -0.5))

    def forward(self, s: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        return self.net(s), self.log_std


def select_action(policy: PolicyNet, obs: np.ndarray) -> tuple[np.ndarray, torch.Tensor]:
    """Sample action from Gaussian policy and return (action, log_prob)."""
    s = torch.tensor(obs, dtype=torch.float32, device=device)
    mean, log_std = policy(s)
    std = log_std.exp()
    dist = torch.distributions.Normal(mean, std)
    a = dist.sample()
    log_prob = dist.log_prob(a).sum()
    return a.cpu().numpy(), log_prob


def compute_returns(rewards: list[float], gamma: float = 0.99) -> torch.Tensor:
    """Discounted returns, normalised."""
    G, returns = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32, device=device)
    return (returns - returns.mean()) / (returns.std() + 1e-8)

In [ ]:
env = gym.make('Hopper-v5')
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]

policy = PolicyNet(obs_dim, act_dim).to(device)
optimizer = optim.Adam(policy.parameters(), lr=3e-4)

ep_returns: list[float] = []

for ep in range(1000):
    obs, _ = env.reset()
    log_probs, rewards = [], []

    while True:
        a, lp = select_action(policy, obs)
        obs, r, terminated, truncated, _ = env.step(a)
        log_probs.append(lp)
        rewards.append(r)
        if terminated or truncated:
            break

    returns = compute_returns(rewards)
    loss = -torch.stack(log_probs) @ returns / len(returns)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    ep_returns.append(sum(rewards))
    if ep % 100 == 0:
        print(f'ep {ep:4d}  return={ep_returns[-1]:.1f}')

env.close()
print('Done.')

In [ ]:
def smooth(x: list[float], w: int = 20) -> np.ndarray:
    return np.convolve(x, np.ones(w) / w, mode='valid')

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=ep_returns, mode='lines',
    line=dict(color='rgba(0,212,255,0.25)', width=1),
    name='raw',
))
fig.add_trace(go.Scatter(
    y=smooth(ep_returns), mode='lines',
    line=dict(color='rgba(0,212,255,1)', width=2),
    name='smoothed',
))
fig.update_layout(
    title='REINFORCE — Hopper-v5',
    xaxis_title='episode', yaxis_title='return',
    template='plotly_dark', height=350,
)
fig.show()

In [ ]:
# Render one greedy episode
env_render = gym.make('Hopper-v5', render_mode='rgb_array')
obs, _ = env_render.reset(seed=0)
frames = [env_render.render()]

policy.eval()
with torch.no_grad():
    done = False
    while not done:
        s = torch.tensor(obs, dtype=torch.float32, device=device)
        mean, _ = policy(s)
        obs, _, terminated, truncated, _ = env_render.step(mean.cpu().numpy())
        frames.append(env_render.render())
        done = terminated or truncated
env_render.close()

fig_r, ax = plt.subplots(figsize=(5, 4))
ax.axis('off')
im = ax.imshow(frames[0])
def update(i):
    im.set_data(frames[i])
    return (im,)
ani = animation.FuncAnimation(fig_r, update, frames=len(frames), interval=30, blit=True)
plt.close(fig_r)
HTML(ani.to_jshtml())

## Part 2 — PPO on Hopper

**Clipped surrogate** (sampled from old policy $\pi_{\theta_\text{old}}$):

$$\mathcal{L}(\theta) = \underset{\substack{S_t \sim d^{\pi_{\theta_\text{old}}} \\ A_t \sim \pi_{\theta_\text{old}}(\cdot \mid S_t)}}{\mathbb{E}}\!\left[\min\!\left(r_t(\theta)\,\hat{A}_t,\;\operatorname{clip}(r_t(\theta),\,1\!\pm\!\varepsilon)\,\hat{A}_t\right)\right]$$

where $r_t(\theta) = \dfrac{\pi_\theta(A_t \mid S_t)}{\pi_{\theta_\text{old}}(A_t \mid S_t)}$ and $\hat{A}_t$ is estimated via **GAE**.

Key: **8 vectorised environments** via `gymnasium.vector.SyncVectorEnv`.

In [ ]:
# ── Actor-Critic ───────────────────────────────────────────────────────────────
class ActorCritic(nn.Module):
    def __init__(self, obs_dim: int, act_dim: int, hidden: int = 64):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.mean_head  = nn.Linear(hidden, act_dim)
        self.value_head = nn.Linear(hidden, 1)
        self.log_std    = nn.Parameter(torch.zeros(act_dim))

    def forward(self, s: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        x = self.shared(s)
        return self.mean_head(x), self.value_head(x).squeeze(-1), self.log_std

In [ ]:
def compute_gae(
    rewards: np.ndarray,
    values: np.ndarray,
    dones: np.ndarray,
    gamma: float = 0.99,
    lam: float = 0.95,
) -> tuple[np.ndarray, np.ndarray]:
    """GAE over (T, N) arrays."""
    T, N = rewards.shape
    advantages = np.zeros((T, N), dtype=np.float32)
    gae = np.zeros(N, dtype=np.float32)
    for t in reversed(range(T)):
        next_val = values[t + 1] if t < T - 1 else np.zeros(N, dtype=np.float32)
        mask = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_val * mask - values[t]
        gae = delta + gamma * lam * mask * gae
        advantages[t] = gae
    returns = advantages + values
    return advantages, returns

In [ ]:
def collect_rollouts(
    ac: ActorCritic,
    envs: gym.vector.VectorEnv,
    obs: np.ndarray,
    n_steps: int = 2048,
) -> tuple[dict, np.ndarray]:
    """Collect n_steps from vectorised envs."""
    obs_buf, act_buf, rew_buf, done_buf, val_buf, logp_buf = [], [], [], [], [], []

    for _ in range(n_steps):
        s = torch.tensor(obs, dtype=torch.float32, device=device)
        with torch.no_grad():
            mean, val, log_std = ac(s)
            std = log_std.exp()
            dist = torch.distributions.Normal(mean, std)
            a = dist.sample()
            lp = dist.log_prob(a).sum(-1)

        obs_buf.append(obs.astype(np.float32))
        act_buf.append(a.cpu().numpy().astype(np.float32))
        val_buf.append(val.cpu().numpy().astype(np.float32))
        logp_buf.append(lp.cpu().numpy().astype(np.float32))

        obs, rew, terminated, truncated, _ = envs.step(a.cpu().numpy())
        done = np.logical_or(terminated, truncated).astype(np.float32)
        rew_buf.append(rew.astype(np.float32))
        done_buf.append(done)

    return dict(
        obs=np.array(obs_buf),
        actions=np.array(act_buf),
        rewards=np.array(rew_buf),
        dones=np.array(done_buf),
        values=np.array(val_buf),
        log_probs_old=np.array(logp_buf),
    ), obs

In [ ]:
def ppo_update(
    ac: ActorCritic,
    optimizer: optim.Optimizer,
    batch: dict,
    advantages: np.ndarray,
    returns: np.ndarray,
    clip_eps: float = 0.2,
    c1: float = 0.5,
    c2: float = 0.01,
    n_epochs: int = 10,
    batch_size: int = 256,
) -> float:
    """Run multiple PPO epochs on the collected batch."""
    obs_f    = torch.tensor(batch['obs'].reshape(-1, batch['obs'].shape[-1]),    dtype=torch.float32, device=device)
    act_f    = torch.tensor(batch['actions'].reshape(-1, batch['actions'].shape[-1]), dtype=torch.float32, device=device)
    old_lp_f = torch.tensor(batch['log_probs_old'].reshape(-1), dtype=torch.float32, device=device)
    adv_f    = torch.tensor(advantages.reshape(-1), dtype=torch.float32, device=device)
    ret_f    = torch.tensor(returns.reshape(-1),    dtype=torch.float32, device=device)

    adv_f = (adv_f - adv_f.mean()) / (adv_f.std() + 1e-8)
    N = len(obs_f)

    total_loss = 0.0
    for _ in range(n_epochs):
        idx = torch.randperm(N, device=device)
        for start in range(0, N, batch_size):
            mb = idx[start:start + batch_size]
            mean, val, log_std = ac(obs_f[mb])
            std = log_std.exp()
            dist = torch.distributions.Normal(mean, std)
            new_lp = dist.log_prob(act_f[mb]).sum(-1)

            ratio = (new_lp - old_lp_f[mb]).exp()
            adv_mb = adv_f[mb]
            l_clip = torch.min(
                ratio * adv_mb,
                ratio.clamp(1 - clip_eps, 1 + clip_eps) * adv_mb,
            ).mean()
            l_value   = ((val - ret_f[mb]) ** 2).mean()
            l_entropy = dist.entropy().sum(-1).mean()

            loss = -(l_clip - c1 * l_value + c2 * l_entropy)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(ac.parameters(), 0.5)
            optimizer.step()
            total_loss += loss.item()

    return total_loss

In [ ]:
N_ENVS    = 8
N_STEPS   = 256      # steps per env per iteration
N_ITERS   = 200

vec_env = gym.vector.SyncVectorEnv([lambda: gym.make('Hopper-v5') for _ in range(N_ENVS)])
obs_dim = vec_env.single_observation_space.shape[0]
act_dim = vec_env.single_action_space.shape[0]

ac = ActorCritic(obs_dim, act_dim).to(device)
ppo_optim = optim.Adam(ac.parameters(), lr=3e-4)

obs, _ = vec_env.reset()
iter_returns: list[float] = []

for it in range(N_ITERS):
    batch, obs = collect_rollouts(ac, vec_env, obs, N_STEPS)
    advantages, returns = compute_gae(batch['rewards'], batch['values'], batch['dones'])
    ppo_update(ac, ppo_optim, batch, advantages, returns)

    mean_return = batch['rewards'].sum(0).mean()
    iter_returns.append(float(mean_return))
    if it % 20 == 0:
        print(f'iter {it:4d}  mean_return={mean_return:.1f}')

vec_env.close()
print('Done.')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=iter_returns, mode='lines',
    line=dict(color='rgba(34,197,94,0.3)', width=1),
    name='raw',
))
fig.add_trace(go.Scatter(
    y=smooth(iter_returns, w=10), mode='lines',
    line=dict(color='rgba(34,197,94,1)', width=2),
    name='smoothed',
))
fig.update_layout(
    title='PPO — Hopper-v5 (8 parallel envs)',
    xaxis_title='iteration', yaxis_title='mean return per env',
    template='plotly_dark', height=350,
)
fig.show()

In [ ]:
# Render one greedy episode of the trained Hopper
env_render = gym.make('Hopper-v5', render_mode='rgb_array')
obs, _ = env_render.reset(seed=0)
frames = [env_render.render()]

ac.eval()
with torch.no_grad():
    done = False
    while not done:
        s = torch.tensor(obs, dtype=torch.float32, device=device)
        mean, _, _ = ac(s)
        obs, _, terminated, truncated, _ = env_render.step(mean.cpu().numpy())
        frames.append(env_render.render())
        done = terminated or truncated
env_render.close()

fig_r, ax = plt.subplots(figsize=(5, 4))
ax.axis('off')
im = ax.imshow(frames[0])
def update(i):
    im.set_data(frames[i])
    return (im,)
ani = animation.FuncAnimation(fig_r, update, frames=len(frames), interval=30, blit=True)
plt.close(fig_r)
HTML(ani.to_jshtml())